# AI Court System Prototype (Canopy Wave)

Three model roles, all served by Canopy Wave's OpenAI-compatible inference API:

- **Simple Counsel** — fast, direct read of the case
- **Complex Counsel** — deep analysis: precedent, counterarguments, edge cases
- **Judge** — receives both counsel opinions and issues the final verdict

Flow: `case (context + query) -> [simple, complex] -> judge -> final verdict`

This notebook is the prototype behind the `services/` microservices (gateway + court-orchestrator).


In [ ]:
# If needed:
# %pip install -U openai python-dotenv pandas httpx

import os
import time
import json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

from openai import OpenAI

def _normalize_secret(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    return value.strip().strip('"').strip("'")

def _mask_secret(value: str) -> str:
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"

def _manual_read_env_key(env_path: Path, key: str) -> Optional[str]:
    if not env_path.exists():
        return None
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        left, right = line.split("=", 1)
        if left.strip() == key:
            return _normalize_secret(right)
    return None

cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd.parent / ".env",
    cwd.parent / "Demo" / ".env",
    cwd / "Demo" / ".env",
]

try:
    from dotenv import load_dotenv
    for candidate in env_candidates:
        if candidate.exists():
            load_dotenv(dotenv_path=candidate, override=False)
except Exception:
    pass

API_KEY = _normalize_secret(os.environ.get("CANOPYWAVE_API_KEY"))
key_source = "environment" if API_KEY else None

if not API_KEY:
    for candidate in env_candidates:
        key_from_file = _manual_read_env_key(candidate, "CANOPYWAVE_API_KEY")
        if key_from_file:
            API_KEY = key_from_file
            key_source = str(candidate)
            break

if not API_KEY:
    checked = ", ".join(str(path) for path in env_candidates)
    raise ValueError("Missing CANOPYWAVE_API_KEY. Checked env and: " + checked)

BASE_URL = _normalize_secret(os.environ.get("CANOPYWAVE_BASE_URL")) or "https://inference.canopywave.io/v1"
BASE_URL = BASE_URL.rstrip("/")

if not BASE_URL.startswith("http") or "/v1" not in BASE_URL:
    raise ValueError("Invalid CANOPYWAVE_BASE_URL, expected .../v1")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f"Base URL: {BASE_URL}")
print(f"API key source: {key_source or 'unknown'}")
print(f"API key preview: {_mask_secret(API_KEY)}")
print("Client initialized successfully")

In [ ]:
def fetch_supported_model_ids() -> List[str]:
    model_page = client.models.list()
    return sorted({m.id for m in model_page.data})

SUPPORTED_MODELS = fetch_supported_model_ids()
print(f"Supported models: {len(SUPPORTED_MODELS)}")
for mid in SUPPORTED_MODELS:
    print(mid)

In [ ]:
# Only models the current Canopy Wave key has inference access to.
# The original prototype cast (kimi-k2.7-code-highspeed / kimi-k3 / mimo-v2.5)
# is listed by /models but returns 403 for this account.
ROLE_MODELS = {
    "simple": "moonshotai/kimi-k2.6",
    "complex": "minimax/minimax-m3",
    "judge": "minimax/minimax-m3",
}

missing_models = [m for m in ROLE_MODELS.values() if m not in SUPPORTED_MODELS]
if missing_models:
    raise ValueError(f"Unsupported ROLE_MODELS entries: {missing_models}")

ROLE_INSTRUCTIONS = {
    "simple": (
        "You are Simple Counsel in an AI court. Argue the case plainly and quickly, "
        "focusing on the most obvious facts and the most direct reading of the dispute."
    ),
    "complex": (
        "You are Complex Counsel in an AI court. Analyze the case in depth: weigh "
        "precedent, counterarguments, edge cases, and second-order consequences before "
        "reaching a position."
    ),
    "judge": (
        "You are the Judge in an AI court. You are given the case plus the structured "
        "opinions of Simple Counsel and Complex Counsel. Weigh both, resolve their "
        "disagreements explicitly, and issue a final ruling."
    ),
}

INFERENCE_CONFIG = {
    "temperature": 0.2,
    "max_tokens": 2000,
    "timeout_seconds": 45,
    "max_retries": 3,
    "backoff_base_seconds": 1.5,
}

print(json.dumps(ROLE_MODELS, indent=2))

In [ ]:
@dataclass
class ModelCallResult:
    role: str
    model: str
    answer: str
    verdict: str
    confidence: Optional[float]
    rationale: str
    latency_ms: int
    retries_used: int
    status: str
    error: Optional[str] = None

@dataclass
class CaseResult:
    case_id: str
    title: str
    simple_verdict: str
    complex_verdict: str
    final_verdict: str
    judge_rationale: str
    simple_latency_ms: int
    complex_latency_ms: int
    judge_latency_ms: int
    total_latency_ms: int
    total_retries: int
    status: str
    error: Optional[str] = None

def _extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Parse the first balanced JSON object found in text (handles markdown
    fences, reasoning preambles, and trailing commentary)."""
    decoder = json.JSONDecoder()
    idx = text.find("{")
    while idx != -1:
        try:
            payload, _ = decoder.raw_decode(text[idx:])
            if isinstance(payload, dict):
                return payload
        except Exception:
            pass
        idx = text.find("{", idx + 1)
    return None

def parse_structured_response(text: str) -> Dict[str, Any]:
    verdict = "UNKNOWN"
    confidence = None
    rationale = ""
    cleaned = (text or "").strip()
    if not cleaned:
        return {"answer": "", "verdict": verdict, "confidence": confidence, "rationale": rationale}

    payload = _extract_json_object(cleaned)
    if payload is not None:
        verdict = str(payload.get("verdict", verdict)).upper()
        confidence_raw = payload.get("confidence")
        try:
            confidence = float(confidence_raw) if confidence_raw is not None else None
        except (TypeError, ValueError):
            confidence = None
        rationale = str(payload.get("rationale", ""))
        return {"answer": cleaned, "verdict": verdict, "confidence": confidence, "rationale": rationale}

    lowered = cleaned.lower()
    if "plaintiff" in lowered:
        verdict = "PLAINTIFF"
    elif "defendant" in lowered:
        verdict = "DEFENDANT"
    return {"answer": cleaned, "verdict": verdict, "confidence": confidence, "rationale": cleaned}

def call_role_model(role: str, user_prompt: str) -> ModelCallResult:
    model_name = ROLE_MODELS[role]
    retries_used = 0
    last_error = None
    system_prompt = (
        ROLE_INSTRUCTIONS[role]
        + " Return strict JSON with keys: verdict, confidence, rationale. "
        + "verdict must be one of: PLAINTIFF, DEFENDANT, MIXED, UNKNOWN. "
        + "confidence must be a number between 0 and 1."
    )

    for attempt in range(INFERENCE_CONFIG["max_retries"] + 1):
        started = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=INFERENCE_CONFIG["temperature"],
                max_tokens=INFERENCE_CONFIG["max_tokens"],
                timeout=INFERENCE_CONFIG["timeout_seconds"],
            )
            raw_text = response.choices[0].message.content if response.choices else ""
            parsed = parse_structured_response(raw_text)
            elapsed_ms = int((time.perf_counter() - started) * 1000)
            return ModelCallResult(
                role=role,
                model=model_name,
                answer=parsed["answer"],
                verdict=parsed["verdict"],
                confidence=parsed["confidence"],
                rationale=parsed["rationale"],
                latency_ms=elapsed_ms,
                retries_used=retries_used,
                status="ok",
            )
        except Exception as exc:
            elapsed_ms = int((time.perf_counter() - started) * 1000)
            last_error = str(exc)
            if attempt < INFERENCE_CONFIG["max_retries"]:
                retries_used += 1
                backoff = INFERENCE_CONFIG["backoff_base_seconds"] ** retries_used
                print(f"[{role}] retry {retries_used}/{INFERENCE_CONFIG['max_retries']} after error: {last_error[:140]}")
                time.sleep(backoff)
            else:
                return ModelCallResult(
                    role=role,
                    model=model_name,
                    answer="",
                    verdict="UNKNOWN",
                    confidence=None,
                    rationale="",
                    latency_ms=elapsed_ms,
                    retries_used=retries_used,
                    status="error",
                    error=last_error,
                )

    return ModelCallResult(
        role=role,
        model=model_name,
        answer="",
        verdict="UNKNOWN",
        confidence=None,
        rationale="",
        latency_ms=0,
        retries_used=retries_used,
        status="error",
        error=last_error or "Unknown error",
    )

## Court orchestration

Both counsels review the case, then the judge rules with their opinions in hand.


In [ ]:
import uuid

def build_case_prompt(title: str, context: str, query: str) -> str:
    parts = [f"Case Title: {title}"]
    if context.strip():
        parts.append(f"Case Facts / Context:\n{context.strip()}")
    if query.strip():
        parts.append(f"Question before the court:\n{query.strip()}")
    return "\n\n".join(parts)

def build_judge_prompt(title: str, context: str, query: str,
                       simple: ModelCallResult, complex_: ModelCallResult) -> str:
    def opinion_block(label: str, result: ModelCallResult) -> str:
        if result.status != "ok":
            return f"{label}: UNAVAILABLE (error: {result.error})"
        return (
            f"{label} (model {result.model}):\n"
            f"  verdict: {result.verdict}\n"
            f"  confidence: {result.confidence}\n"
            f"  rationale: {result.rationale or result.answer}"
        )

    return (
        build_case_prompt(title, context, query)
        + "\n\n--- Counsel Opinions ---\n\n"
        + opinion_block("Simple Counsel", simple)
        + "\n\n"
        + opinion_block("Complex Counsel", complex_)
        + "\n\nIssue your final ruling."
    )

def run_case(title: str, context: str = "", query: str = "") -> CaseResult:
    started = time.perf_counter()
    case_prompt = build_case_prompt(title, context, query)

    simple_result = call_role_model("simple", case_prompt)
    complex_result = call_role_model("complex", case_prompt)
    judge_result = call_role_model(
        "judge", build_judge_prompt(title, context, query, simple_result, complex_result)
    )

    total_latency_ms = int((time.perf_counter() - started) * 1000)
    role_results = [simple_result, complex_result, judge_result]
    errors = [f"{r.role}: {r.error}" for r in role_results if r.status == "error"]

    return CaseResult(
        case_id=uuid.uuid4().hex[:12],
        title=title,
        simple_verdict=simple_result.verdict,
        complex_verdict=complex_result.verdict,
        final_verdict=judge_result.verdict,
        judge_rationale=judge_result.rationale or judge_result.answer,
        simple_latency_ms=simple_result.latency_ms,
        complex_latency_ms=complex_result.latency_ms,
        judge_latency_ms=judge_result.latency_ms,
        total_latency_ms=total_latency_ms,
        total_retries=sum(r.retries_used for r in role_results),
        status="error" if judge_result.status == "error" else "ok",
        error="; ".join(errors) if errors else None,
    )

## Demo cases

Run a few sample cases through the full court and summarize the docket.


In [ ]:
DEMO_CASES = [
    {
        "title": "Orchard Lane Fence Dispute",
        "context": (
            "The plaintiff hired the defendant to build a cedar fence for $4,200, half paid up front. "
            "The contract specified completion within 30 days. The defendant finished on day 52, "
            "citing two weeks of heavy rain documented by local weather records. The plaintiff "
            "refuses to pay the remaining balance and claims $600 in damages for a garden party "
            "that had to be relocated."
        ),
        "query": "Is the defendant entitled to the remaining balance, and does the plaintiff have a valid damages claim?",
    },
    {
        "title": "Freelance Logo Ownership",
        "context": (
            "A startup (plaintiff) paid a freelance designer (defendant) $1,500 for a logo. "
            "No written contract addressed IP transfer. After delivery, the designer reused core "
            "elements of the logo for another client in a different industry. The startup demands "
            "exclusive rights and removal of the second logo."
        ),
        "query": "Who owns the logo design, and did the designer breach any obligation by reusing elements?",
    },
]

docket_results = []
for demo_case in DEMO_CASES:
    print(f"Running case: {demo_case['title']} ...")
    result = run_case(demo_case["title"], demo_case["context"], demo_case["query"])
    docket_results.append(result)
    print(f"  simple={result.simple_verdict} complex={result.complex_verdict} "
          f"final={result.final_verdict} ({result.total_latency_ms} ms)\n")

In [ ]:
import pandas as pd

summary_df = pd.DataFrame([asdict(r) for r in docket_results])
display_cols = [
    "case_id", "title", "simple_verdict", "complex_verdict", "final_verdict",
    "simple_latency_ms", "complex_latency_ms", "judge_latency_ms",
    "total_latency_ms", "total_retries", "status",
]
summary_df[display_cols]

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
export_path = results_dir / "docket_results.jsonl"

with export_path.open("w", encoding="utf-8") as fh:
    for r in docket_results:
        fh.write(json.dumps(asdict(r)) + "\n")

print(f"Exported {len(docket_results)} case results to {export_path.resolve()}")

for r in docket_results:
    print(f"\n=== {r.title} -> {r.final_verdict} ===")
    print(r.judge_rationale[:600])